# Lane Following Lab: Rules vs. Learning

In the lecture you saw two ways to make a computer process data. You can write the rules by hand, or you can give an algorithm examples of inputs and correct outputs and let it find the rules itself.

In this lab you will do both, on the same problem, and compare.

The problem: a small robot drives along a lane at constant speed. It has a tiny black and white camera, and the only thing it controls is steering. Your job is to turn the camera image into a steering command.

**Part 1, The Robot.** You meet the robot, look through its camera, and paint your first steering rule.

**Part 2, Challenge 1: The Hand-Written Driver.** You write rules that keep the robot in the lane on five different tracks.

**Part 3, Learning the Rules.** You let an algorithm learn a driver from recorded examples and compare it with yours.

**Part 4, Challenge 2: The Data Designer.** You improve the learned driver without touching a single weight. You only choose what the instructor records.

**Part 5, New Robots.** The camera changes, twice. You see what that costs your rules, and what it costs the learner.

How to work: exercise cells contain `TODO` markers. A check cell tells you whether your code behaves correctly, and a solution follows below it. Write your own attempt first, then compare. Challenges have no single right answer: you iterate and push your score as low as you can.

In [ ]:
# Load the tools for this lab. numpy does the math, matplotlib draws the
# pictures, scipy finds the nearest point of the lane quickly.

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

print("Ready.")

## Part 1: The Robot

The robot drives on a lane that is 60 cm wide and marked by two painted lines. It always drives at 0.5 m/s. Every 50 ms it takes a camera picture and you decide the steering:

- **positive steering turns left**, negative steering turns right
- steering is a turning rate in rad/s, limited to between -3 and +3
- if the robot's center leaves the lane, the run is over

The next three cells build the world: the tracks, the camera, and a few tools to look at things. They are collapsed, just run them. You do not need to read them to do the lab.

In [ ]:
# @title The world: lanes and tracks (run it, no need to read it)

LANE_WIDTH = 0.6     # meters between the two painted lines
LINE_WIDTH = 0.07    # width of one painted line
SPEED = 0.5          # the robot always drives at 0.5 m/s
DT = 0.05            # one simulation step is 50 ms
MAX_STEER = 3.0      # steering limit in rad/s, positive turns left


class Track:
    def __init__(self, name, centerline, width=LANE_WIDTH, dashed=False, glare=0.0):
        self.name = name
        self.xy = centerline
        self.width = np.zeros(len(centerline)) + width
        self.dashed = dashed
        self.glare = glare

        # Direction of the lane, and the "left" direction, at every point
        step = np.roll(centerline, -1, axis=0) - centerline
        seg = np.linalg.norm(step, axis=1)
        self.tangent = step / seg[:, None]
        self.normal = np.stack([-self.tangent[:, 1], self.tangent[:, 0]], axis=1)

        # Distance along the lane, and a fast nearest point lookup
        self.s = np.concatenate([[0.0], np.cumsum(seg)[:-1]])
        self.length = seg.sum()
        self.tree = cKDTree(centerline)

    def locate(self, points):
        # Nearest lane point for each point, and the signed distance from
        # the lane center (positive means left of the center)
        _, i = self.tree.query(points)
        offset = np.einsum("ij,ij->i", points - self.xy[i], self.normal[i])
        return i, offset


def stadium(straight, radius, n=3000):
    # Two straights joined by two half circles, driven counterclockwise
    arc = np.pi * radius
    half = straight / 2
    s = np.linspace(0, 2 * straight + 2 * arc, n, endpoint=False)
    pts = np.zeros((n, 2))

    a = s < straight
    pts[a] = np.stack([s[a] - half, np.full(a.sum(), -radius)], axis=1)

    b = (s >= straight) & (s < straight + arc)
    ang = -np.pi / 2 + (s[b] - straight) / radius
    pts[b] = np.stack([half + radius * np.cos(ang), radius * np.sin(ang)], axis=1)

    c = (s >= straight + arc) & (s < 2 * straight + arc)
    pts[c] = np.stack([half - (s[c] - straight - arc), np.full(c.sum(), radius)], axis=1)

    d = s >= 2 * straight + arc
    ang = np.pi / 2 + (s[d] - 2 * straight - arc) / radius
    pts[d] = np.stack([-half + radius * np.cos(ang), radius * np.sin(ang)], axis=1)

    return pts


def wavy_loop(radius, wiggle, lobes, phase=0.0, n=3000):
    # A closed loop whose radius goes in and out, giving left and right bends
    phi = np.linspace(0, 2 * np.pi, n, endpoint=False)
    r = radius * (1 + wiggle * np.sin(lobes * phi + phase))
    return np.stack([r * np.cos(phi), r * np.sin(phi)], axis=1)


phi = np.linspace(0, 2 * np.pi, 3000, endpoint=False)

TRACKS = {
    "oval":    Track("oval", stadium(3.0, 1.2)),
    "curves":  Track("curves", wavy_loop(2.2, 0.30, 3)),
    "dashed":  Track("dashed", wavy_loop(2.0, 0.20, 2, np.pi / 2), dashed=True),
    "glare":   Track("glare", stadium(2.5, 1.0), glare=0.08),
    "squeeze": Track("squeeze", wavy_loop(2.1, 0.15, 2, 1.0), width=LANE_WIDTH + 0.2 * np.sin(5 * phi)),
}

print("Tracks:", ", ".join(TRACKS))

In [ ]:
# @title The robot and its camera (run it, no need to read it)

ROWS, COLS = 10, 16

# Where every camera pixel looks, in meters, measured from the robot.
# FORWARD: how far ahead (top row is far, bottom row is near).
# LATERAL: how far to the left (left columns are positive).
FORWARD = np.repeat(np.linspace(1.0, 0.1, ROWS)[:, None], COLS, axis=1)
LATERAL = np.repeat(np.linspace(0.4, -0.4, COLS)[None, :], ROWS, axis=0)

MOUNT = {"shift": 0.0, "yaw": 0.0, "wiring": None}   # how the camera sits on the robot


def camera(track, x, y, heading, rng=None):
    # Where each pixel really looks relative to the robot
    c, s = np.cos(MOUNT["yaw"]), np.sin(MOUNT["yaw"])
    fwd = c * FORWARD - s * LATERAL
    left = s * FORWARD + c * LATERAL + MOUNT["shift"]

    # ... and where that is in the world
    c, s = np.cos(heading), np.sin(heading)
    px = x + c * fwd - s * left
    py = y + s * fwd + c * left
    i, offset = track.locate(np.stack([px.ravel(), py.ravel()], axis=1))

    # A pixel is white if it lands on one of the two painted lines
    half = track.width[i] / 2
    on_left = np.abs(offset - half) < LINE_WIDTH / 2
    on_right = np.abs(offset + half) < LINE_WIDTH / 2
    if track.dashed:
        on_left &= (track.s[i] % 0.45) < 0.30
    image = (on_left | on_right).astype(float).reshape(ROWS, COLS)

    # Sun glare: random white pixels
    if track.glare > 0 and rng is not None:
        image = np.maximum(image, rng.random(image.shape) < track.glare)

    # A wrongly plugged cable delivers the pixels in a different order
    if MOUNT["wiring"] is not None:
        image = image.ravel()[MOUNT["wiring"]].reshape(ROWS, COLS)

    return image


def pose_on_track(track, at_m=0.0, offset=0.0, heading=0.0):
    # A robot pose: at_m meters along the lane, offset meters left of the
    # center, heading radians turned left relative to the lane
    i = min(np.searchsorted(track.s, at_m % track.length), len(track.xy) - 1)
    x, y = track.xy[i] + offset * track.normal[i]
    lane_heading = np.arctan2(track.tangent[i, 1], track.tangent[i, 0])
    return x, y, lane_heading + heading


def instructor(track, x, y, heading, lookahead=0.6):
    # A driver who knows exactly where the lane is. It aims at the lane
    # center 0.6 m ahead and steers along a circle that hits that point.
    i, _ = track.locate(np.array([[x, y]]))
    j = (i[0] + int(lookahead / track.length * len(track.xy))) % len(track.xy)

    dx, dy = track.xy[j] - (x, y)
    ahead = np.cos(heading) * dx + np.sin(heading) * dy
    left = -np.sin(heading) * dx + np.cos(heading) * dy

    return 2 * SPEED * left / (ahead ** 2 + left ** 2)


def drive(driver, track_name, seed=0):
    # Drive one lap. The driver only ever sees the camera image.
    track = TRACKS[track_name]
    rng = np.random.default_rng(seed)
    x, y, heading = pose_on_track(track)
    last, _ = track.locate(np.array([[x, y]]))
    last = last[0]

    progress, offsets, path = 0.0, [], [(x, y)]
    status, image = "too slow", None
    n = len(track.xy)

    for _ in range(int(2 * track.length / SPEED / DT)):
        # Look, decide, move
        image = camera(track, x, y, heading, rng)
        steer = float(np.clip(driver(image), -MAX_STEER, MAX_STEER))
        x += SPEED * np.cos(heading) * DT
        y += SPEED * np.sin(heading) * DT
        heading += steer * DT
        path.append((x, y))

        # Measure progress along the lane and distance from its center
        i, offset = track.locate(np.array([[x, y]]))
        i, offset = i[0], offset[0]
        progress += ((i - last + n // 2) % n - n // 2) * track.length / n
        last = i
        offsets.append(abs(offset))

        if abs(offset) > track.width[i] / 2:
            status = "left the lane"
            break
        if progress >= track.length:
            status = "lap complete"
            break

    return {
        "status": status,
        "percent": float(np.clip(100 * progress / track.length, 0, 100)),
        "cm_from_center": 100 * float(np.mean(offsets)),
        "path": np.array(path),
        "last_image": image,
    }


def record_demonstrations(track_names=None, laps=2, wobble=2.4, seed=0):
    # The instructor drives every track. Its own steering wobbles on purpose,
    # so the data also shows how to recover from mistakes. We store every
    # camera image and the steering the instructor wanted at that moment.
    # wobble sets how strongly its hands shake, 0 is a perfect driver.
    rng = np.random.default_rng(seed)
    images, steering = [], []

    for name in track_names or TRACKS:
        track = TRACKS[name]
        for _ in range(laps):
            x, y, heading = pose_on_track(track, rng.random() * track.length)
            shake = 0.0

            for _ in range(int(track.length / SPEED / DT)):
                images.append(camera(track, x, y, heading, rng).ravel())
                wanted = instructor(track, x, y, heading)
                steering.append(wanted)

                shake = 0.9 * shake + 0.1 * rng.normal(0, wobble)
                steer = np.clip(wanted + shake, -MAX_STEER, MAX_STEER)
                x += SPEED * np.cos(heading) * DT
                y += SPEED * np.sin(heading) * DT
                heading += steer * DT

                # Too far off? Put the robot back on the center line.
                i, offset = track.locate(np.array([[x, y]]))
                if abs(offset[0]) > 0.45 * track.width[i[0]]:
                    x, y, heading = pose_on_track(track, track.s[i[0]])

    return np.array(images), np.array(steering)


print("Camera:", ROWS, "rows x", COLS, "columns")

In [ ]:
# @title Tools to look at things (run it, no need to read it)

def draw_track(ax, track):
    for side in (1, -1):
        edge = track.xy + side * (track.width[:, None] / 2) * track.normal
        style = "--" if (track.dashed and side == 1) else "-"
        ax.plot(*np.vstack([edge, edge[:1]]).T, style, color="0.3", lw=1)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])


def draw_image(ax, image, title=""):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.axvline(COLS / 2 - 0.5, color="tab:orange", lw=1)
    ax.set_xticks([0, COLS - 1], ["left", "right"])
    ax.set_yticks([0, ROWS - 1], ["far", "near"])
    ax.set_title(title, fontsize=10)


def show_tracks():
    fig, axes = plt.subplots(1, len(TRACKS), figsize=(15, 3.2))
    for ax, track in zip(axes, TRACKS.values()):
        draw_track(ax, track)
        ax.plot(*track.xy[0], "o", color="tab:green")
        ax.set_title(track.name)
    plt.tight_layout()
    plt.show()


def look(offset_cm=0, heading_deg=0, track="oval", at_m=1.0, driver=None):
    # Park the robot somewhere and look through its camera
    t = TRACKS[track]
    x, y, heading = pose_on_track(t, at_m, offset_cm / 100, np.radians(heading_deg))
    image = camera(t, x, y, heading, np.random.default_rng(0))

    title = f"instructor steers {instructor(t, x, y, heading):+.2f}"
    if driver is not None:
        title += f",  you steer {driver(image):+.2f}"

    fig, ax = plt.subplots(figsize=(4, 2.8))
    draw_image(ax, image, title)
    plt.show()
    return image


def show_drive(driver, track_name):
    # Drive one lap and show the path, and the last camera image
    result = drive(driver, track_name)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.6, 1]})
    draw_track(axes[0], TRACKS[track_name])
    axes[0].plot(*result["path"].T, color="tab:red", lw=1.5)
    axes[0].plot(*result["path"][-1], "x", color="tab:red", ms=12, mew=3)
    axes[0].set_title(f"{track_name}: {result['status']} ({result['percent']:.0f}%)")
    draw_image(axes[1], result["last_image"], "last camera image")
    plt.tight_layout()
    plt.show()


def show_maps(maps):
    # Red pixels push the steering left, blue pixels push it right. The
    # color scale ignores a few extreme pixels so the pattern stays visible.
    fig, axes = plt.subplots(1, len(maps), figsize=(4.2 * len(maps), 2.8))
    for ax, (name, weights) in zip(np.atleast_1d(axes), maps.items()):
        top = np.percentile(np.abs(weights), 95) or 1.0
        ax.imshow(weights, cmap="RdBu_r", vmin=-top, vmax=top)
        ax.axvline(COLS / 2 - 0.5, color="0.5", lw=1)
        ax.set_xticks([0, COLS - 1], ["left", "right"])
        ax.set_yticks([0, ROWS - 1], ["far", "near"])
        ax.set_title(name, fontsize=10)
    plt.tight_layout()
    plt.show()


show_tracks()

### Through the camera

The camera sees 10 rows by 16 columns. A white pixel is a painted line, a black pixel is road.

- the top row looks 1 m ahead, the bottom row 10 cm ahead
- the columns cover 40 cm to the left and 40 cm to the right of the robot
- the orange line marks straight ahead

`look()` parks the robot somewhere, shows its camera image, and tells you how an instructor who knows exactly where the lane is would steer. Here the robot sits in the middle of the oval's straight, pointing straight ahead.

In [ ]:
# The robot in the center of the lane, pointing along it.

image = look(offset_cm=0, heading_deg=0)

### Exercise 1: Pixels times weights

Our driver works exactly like Rosenblatt's perceptron from the lecture. Every pixel gets a weight. The steering command is the sum of every pixel times its weight:

$$\text{steering} = \sum_{\text{pixels}} \text{pixel} \times \text{weight}$$

The weights form a **weight map**, an array with the same shape as the image. A positive weight means "if this pixel is white, steer left". A negative weight means "steer right".

Implement `steer_from_map(image, weights)`.

In [ ]:
# Exercise 1. Fill in the TODO. image and weights are both numpy arrays
# of shape (ROWS, COLS). Useful pieces: a * b multiplies two arrays element
# by element, np.sum(a) adds up all elements of an array.

def steer_from_map(image, weights):
    # TODO: multiply every pixel by its weight and add everything up
    steering = 0.0
    return steering

In [ ]:
# Run this cell to check your function.

test_image = np.zeros((ROWS, COLS))
test_image[9, 2] = 1
test_image[5, 12] = 1

test_weights = np.zeros((ROWS, COLS))
test_weights[9, 2] = 0.5
test_weights[5, 12] = -2.0
test_weights[0, 0] = 7.0

assert abs(steer_from_map(test_image, test_weights) - (-1.5)) < 1e-9, "expected 0.5 - 2.0 = -1.5"
assert steer_from_map(np.zeros((ROWS, COLS)), test_weights) == 0, "a black image should give zero"

print("All checks passed. Pixels times weights.")

**Solution below. Try it yourself first.**

In [ ]:
# Solution for Exercise 1.

def steer_from_map(image, weights):
    return float(np.sum(image * weights))

In [ ]:
# Turn a weight map into a driver: a function from image to steering.
# Nothing to change here.

def map_driver(weights):
    return lambda image: steer_from_map(image, weights)

### Exercise 2: Think before you paint

Before you design a weight map, you need to know what the camera sees when something goes wrong. Use `look()` to put the robot in trouble and study the picture. `offset_cm` moves the robot to the left (negative moves it right), `heading_deg` turns it to the left.

Answer these, with your neighbour if you like:

1. The robot drifts 10 cm to the left. Which pixels change, and in which direction? Which way should it steer?
2. The robot is turned 15 degrees to the left but still in the center. How do the near rows and the far rows differ?
3. The robot drifts 20 cm to the left. One line disappears from the picture. Do you still know where the lane center is?
4. In a single row, where is the lane center, if you can see both lines? If you can see only one?

Check your answers against the instructor's steering.

In [ ]:
# Your playground. Change the numbers and rerun.

image = look(offset_cm=10, heading_deg=0)

In [ ]:
# Your answers.
#
# 1.
# 2.
# 3.
# 4.

### Exercise 3: Paint your first map

Now turn your answers into a weight map. Two arrays help you: `FORWARD` and `LATERAL` have the same shape as the image and tell you where each pixel looks, in meters.

- `LATERAL` is positive on the left, negative on the right, from +0.4 to -0.4
- `FORWARD` is 1.0 in the top row and 0.1 in the bottom row

So `LATERAL > 0` selects the left half of the picture, and `3 * LATERAL` is a map that grows from right to left. Build `my_map` so that:

- a centered robot steers (almost) zero
- a robot that drifted left steers right, and the other way around
- a robot turned to the left steers right

Then drive it on the oval and tune it until it completes a lap.

In [ ]:
# Exercise 3. Fill in the TODO with a weight map of shape (ROWS, COLS).

my_map = np.zeros((ROWS, COLS))   # TODO

In [ ]:
# Run this cell to check your map.

def steer_at(offset_cm, heading_deg):
    t = TRACKS["oval"]
    x, y, heading = pose_on_track(t, 1.0, offset_cm / 100, np.radians(heading_deg))
    return steer_from_map(camera(t, x, y, heading), my_map)


assert abs(steer_at(0, 0)) < 0.05, "a centered robot should barely steer"
assert steer_at(10, 0) < -0.05, "drifted left: it should steer right (negative)"
assert steer_at(-10, 0) > 0.05, "drifted right: it should steer left (positive)"
assert steer_at(0, 15) < -0.05, "turned left: it should steer right (negative)"

print("All checks passed. Now drive it.")

In [ ]:
# Drive one lap of the oval. The red line is the path, the cross marks
# where the run ended. Tune my_map until the lap is complete.

show_drive(map_driver(my_map), "oval")

**Solution below. Try it yourself first.**

In [ ]:
# Solution for Exercise 3. Each pixel pulls the steering toward its own
# side, as strongly as it is far from the middle. Added up over all white
# pixels, this points to the average position of the lines, which is the
# lane center. The factor 2.0 is the gain: how hard to steer.

my_map = 2.0 * LATERAL

show_maps({"Exercise 3 solution": my_map})
show_drive(map_driver(my_map), "oval")

Your map drives the oval. But one oval is not a road network.

## Part 2: Challenge 1, The Hand-Written Driver

Write a driver that keeps the robot in the lane on all five tracks:

- **oval**, the one you know
- **curves**, sharper bends, to the left and to the right
- **dashed**, the left line is dashed
- **glare**, sunlight puts random white pixels in the picture
- **squeeze**, the lane gets narrower and wider

The rules:

- your driver is a function `my_driver(image)` that returns a steering command
- it may only look at the image, nothing else about the world
- inside it you can do anything: weight maps, several maps, `if` and `else`, counting pixels, numpy
- plain `for` loops over rows or pixels are fine, speed does not matter here
- `evaluate` drives one lap on every track and reports how far from the lane center the robot was on average, in cm. A track you do not finish counts as 30 cm. Lower is better.

Scores to beat:

| driver | score |
|---|---|
| the Exercise 3 solution | about 22 |
| the sample solution below | about 6 |
| a really good driver | under 4 |

In [ ]:
# The challenge scorer. The same driver always gets exactly the same
# score, so improvements are real. Nothing to change here.

def evaluate(driver):
    total = 0.0
    print(f"{'track':10}{'result':24}{'cm from center':>15}")

    for name in TRACKS:
        r = drive(driver, name)
        if r["status"] == "lap complete":
            total += r["cm_from_center"]
            print(f"{name:10}{'lap complete':24}{r['cm_from_center']:>15.1f}")
        else:
            total += 30
            result = f"{r['status']} at {r['percent']:.0f}%"
            print(f"{name:10}{result:24}{'30 (penalty)':>15}")

    score = total / len(TRACKS)
    print(f"\nscore: {score:.1f}")
    return score


score = evaluate(map_driver(2.0 * LATERAL))

Two tools help you find out why your driver fails:

- `show_drive(my_driver, "curves")` shows where the robot left the lane, and what the camera saw at that moment
- `look(..., track="curves", at_m=...)` parks the robot anywhere on any track, so you can study a difficult spot at rest

Write your driver below, evaluate, look at the failures, think, improve.

### Numpy toolbox

Your driver works on the image, a numpy array of 10 rows and 16 columns. The cell below shows the handful of tools you need, applied to a real camera image. Run it and read each printed result next to the line that made it.

In [ ]:
# A robot that drifted 15 cm to the left and is turned 10 degrees left.
# Only the left line is in view, and it slants across the picture.

image = look(offset_cm=15, heading_deg=10)

# Element by element math, then add everything up
print("sum of image * LATERAL:  ", np.sum(image * LATERAL))
print("white pixels:            ", image.sum())

# Slicing: image[rows, columns], and ':' means "all of them"
near = image[7:, :]                  # the three nearest rows
left_half = image[:, :COLS // 2]     # the eight left columns
print("white pixels, near rows: ", near.sum())
print("white pixels, left half: ", left_half.sum())

# Masks: True/False arrays that select pixels
print("white pixels left of center:", image[LATERAL > 0].sum())

# np.where(condition, a, b) takes a where the condition is True, else b.
# np.sign gives +1 for positive numbers and -1 for negative ones.
side = np.where(LATERAL > 0, 1.0, -1.0)
print("np.where and np.sign agree:", np.all(side == np.sign(LATERAL)))

# np.nonzero: which columns are white in one row
cols = np.nonzero(image[ROWS - 1])[0]
print("nearest row, white columns:", cols, " at", LATERAL[ROWS - 1, cols], "m")

# A loop over the rows works too
for r in range(ROWS):
    cols = np.nonzero(image[r])[0]
    if len(cols) > 0:
        print(f"row {r}: white pixels {len(cols)}, average position {LATERAL[r, cols].mean():+.2f} m")

In [ ]:
# Challenge 1. Your driver. Start from your map and improve it.

def my_driver(image):
    return steer_from_map(image, my_map)


score = evaluate(my_driver)

In [ ]:
# Your debugging playground.

show_drive(my_driver, "curves")

### Hints

Three ideas, roughly in order of power. Each one is worth trying on its own before combining them.

1. **Look at the crash.** Use `show_drive` on a track that fails and study the last camera image. How many lines does the robot see at that moment? Where does your map think the lane center is?
2. **One line is enough.** The lane is 60 cm wide. If you see a line, you know the lane center is 30 cm to its side. Can a single weight map encode "the center is half a lane away from every line I see"? `np.sign(LATERAL)` tells every pixel which side it is on. Plot your idea with `show_maps` before you drive it.
3. **Near and far.** Near rows tell you where you are now, far rows tell you where the lane is going. Try giving them different weights, and tune the gain again afterwards.

### A sample solution

Below is one decent answer, built on hint 2. It scores about 6. Look at it if you are stuck, or after your own attempt.

Beating it is very possible. Hint 3 can bring it under 4.

In [ ]:
# Sample solution: every white pixel votes for a lane center 30 cm to its
# inner side. A pixel left of the middle says "the center is 30 cm to my
# right", so its weight is LATERAL - 0.3. Right of the middle: LATERAL + 0.3.

sample_map = 2.0 * (LATERAL - 0.3 * np.sign(LATERAL))


def sample_driver(image):
    return steer_from_map(image, sample_map)


show_maps({"sample solution": sample_map})
score = evaluate(sample_driver)

## Part 3: Learning the Rules

Now the other way. Instead of thinking about lines and lane centers, we let an instructor drive. The instructor knows exactly where the lane is and drives two laps on every track. Its hands wobble on purpose, so it also shows how to recover from mistakes.

We record every camera image together with the steering the instructor wanted at that moment. This is the lecture's recipe: **data plus answers in, rules out.**

- `X` holds one image per row, flattened to 160 numbers
- `y` holds the matching steering commands

In [ ]:
# Record the instructor. This takes a few seconds.

X, y = record_demonstrations()

print("X:", X.shape, "  one row per camera image")
print("y:", y.shape, "  one steering command per image")

### Exercise 4: Learn the map

The learned driver has exactly the same form as yours: one weight per pixel, steering is the sum of pixels times weights. Only now nobody paints the weights. We look for the 160 weights `w` that make the predicted steering `X @ w` as close as possible to the instructor's steering `y`, over all recorded images.

This is **least squares**: the same method as fitting a straight line through points, only with 160 numbers instead of two. numpy solves it in one line with `np.linalg.lstsq(X, y, rcond=None)[0]`.

Implement `learn_map(X, y)`. It returns the weights as a map of shape `(ROWS, COLS)`.

In [ ]:
# Exercise 4. Fill in the TODO.

def learn_map(X, y):
    # TODO: find the weights w that make X @ w as close to y as possible
    w = np.zeros(X.shape[1])
    return w.reshape(ROWS, COLS)

In [ ]:
# Run this cell to check your learner.

learned_map = learn_map(X, y)
error = np.sqrt(np.mean((X @ learned_map.ravel() - y) ** 2))

assert learned_map.shape == (ROWS, COLS), "the map should have shape (ROWS, COLS)"
assert error < 0.4, f"predictions are far from the instructor, error {error:.2f}"

print(f"All checks passed. Average steering error: {error:.2f} rad/s")

**Solution below. Try it yourself first.**

In [ ]:
# Solution for Exercise 4.

def learn_map(X, y):
    w = np.linalg.lstsq(X, y, rcond=None)[0]
    return w.reshape(ROWS, COLS)

In [ ]:
# Learn a map from the recordings and run the challenge with it.

learned_map = learn_map(X, y)
score = evaluate(map_driver(learned_map))

### Look inside the learned map

The learner never saw `LATERAL` or `FORWARD`. It does not know what a line, a lane or a center is. It only saw images and steering commands. Compare its map with the ones painted by people.

1. Find the sample solution's logic in the learned map. In which rows is it?
2. What did the learner find that nobody painted? Look at the far rows, especially the corners.
3. The learned map looks noisy. Why does it still drive well?
4. How long did your best hand-written driver take you? How long did the learner need?

In [ ]:
# Painted by people on the left, learned from examples on the right.
# Red pixels push the steering left, blue pixels push it right.

show_maps({
    "Exercise 3 solution": my_map,
    "sample solution": sample_map,
    "learned": learned_map,
})

### Experiment: Fit is not enough

The learner's job was to match the instructor on the recorded images. But that is not what we really want. We want a driver that steers well on images it has **never seen**. The lecture called this **generalization**.

To see the difference we need a second recording that the learner is never allowed to use: `X_test` and `y_test`. Then we let the learner use only `n` of its recorded images, and measure three things:

- the steering error on the images it learned from
- the steering error on the test images it has never seen
- the challenge score

Run the experiment with `n` = 5000, 1000, 320 and 160. Write the numbers down.

1. With 160 images, the error on the learned images is exactly zero. The map has 160 weights. Why can it be perfect?
2. The same map is the worst driver of all. What did it learn?
3. Which of the two errors tells you how well a map will drive?

In [ ]:
# A second recording, only for testing. The learner never sees it.

X_test, y_test = record_demonstrations(seed=1)


def steering_error(weights, X, y):
    # How far the map's steering is from the instructor's, on average
    return np.sqrt(np.mean((X @ weights.ravel() - y) ** 2))


print("Test images:", len(y_test))

In [ ]:
# Learn from only n of the recorded images. Change n and rerun.

n = 5000

picked = np.random.default_rng(1).permutation(len(y))[:n]
small_map = learn_map(X[picked], y[picked])

print(f"learned from {len(picked)} images")
print(f"error on the images it learned from: {steering_error(small_map, X[picked], y[picked]):.2f}")
print(f"error on images it has never seen:   {steering_error(small_map, X_test, y_test):.2f}\n")

score = evaluate(map_driver(small_map))

Fitting the recorded examples is easy. With enough weights you can always do it. The hard part, and the whole point, is doing well on what comes next.

## Part 4: Challenge 2, The Data Designer

In Part 3 you did not write a single rule. But somebody still made decisions: which tracks the instructor drives, for how long, and how much its hands wobble. Those decisions are now yours.

`record_demonstrations` has three settings:

- `track_names`, the list of tracks the instructor drives on
- `laps`, how many laps it drives on each of them
- `wobble`, how strongly its hands shake. 0 is a perfect driver, 2.4 is what Part 3 used.

The rules:

- you may only change these three settings, with at most 4 laps
- the learner stays as it is: `learn_map`, no painting, no editing of weights
- `evaluate` drives all five tracks, as in Challenge 1

The cell below starts with a perfect instructor on the oval. A flawless driver on a clean track: that sounds like the best teacher you could ask for. Run it.

Scores to beat:

| recording | score |
|---|---|
| a perfect instructor on the oval | about 24 |
| the Part 3 recording | about 3 |
| the sample solution below | about 1.8 |
| a really good recording | under 1.6 |

In [ ]:
# Challenge 2. Your recording. Change the three settings and rerun.

X_mine, y_mine = record_demonstrations(
    track_names=["oval"],
    laps=2,
    wobble=0.0,
)

my_learned_map = learn_map(X_mine, y_mine)

print(f"{len(y_mine)} images, error on them: {steering_error(my_learned_map, X_mine, y_mine):.2f}\n")
score = evaluate(map_driver(my_learned_map))

In [ ]:
# Your debugging playground.

show_drive(map_driver(my_learned_map), "curves")

### Hints

Three ideas. Each one is worth trying on its own before combining them.

1. **Look at the crash.** The error on the recorded images is tiny, and still the robot leaves the lane. Use `show_drive` and study the last camera image. Did the perfect instructor ever show the learner a picture like this one? Later on, a score that jumps by about 6 points means that exactly one track was lost. Find out which one, and where.
2. **A track is a lesson.** Record on one track at a time. Which other tracks can the learner drive afterwards? Which track teaches the most? Is there one you can leave out?
3. **Mistakes are lessons too.** Try several values of `wobble`, also small ones. Why is a perfect instructor a bad teacher? Why is a very shaky one not the best either?

### A sample solution

Below is one decent answer. It scores about 1.8. Look at it if you are stuck, or after your own attempt.

In [ ]:
# Sample solution: all five tracks, and an instructor whose hands shake
# only a little. The robot gets to see every kind of road, and it sees
# what the lane looks like from slightly wrong places, together with the
# steering that brings it back. With a large wobble the recording fills up
# with situations far from the lane center, and one map has to fit them all.

X_sample, y_sample = record_demonstrations(
    track_names=["oval", "curves", "dashed", "glare", "squeeze"],
    laps=2,
    wobble=1.0,
)

sample_learned_map = learn_map(X_sample, y_sample)
score = evaluate(map_driver(sample_learned_map))

You improved the driver and never touched a weight. In the learning approach the engineering does not disappear. It moves into the data.

## Part 5: New Robots

The lab got a second robot. Same wheels, same camera, but the mechanic mounted the camera a little differently. Nobody wrote down how.

`use_new_robot()` switches to the new robot. From now on, everything uses its camera: `look`, `show_drive`, `evaluate` and `record_demonstrations`. If you want to go back to an earlier part, `use_old_robot()` switches back.

First, see what happens to the drivers you already have.

In [ ]:
# @title Switch robots (run it, no peeking at the numbers)

def use_new_robot():
    MOUNT.update(shift=0.12, yaw=-0.15, wiring=None)
    print("Now driving the new robot.")


def use_third_robot():
    MOUNT.update(shift=0.0, yaw=0.0, wiring=np.random.default_rng(71).permutation(ROWS * COLS))
    print("Now driving the third robot.")


def use_old_robot():
    MOUNT.update(shift=0.0, yaw=0.0, wiring=None)
    print("Now driving the old robot.")


use_new_robot()

print("\nYour Challenge 1 driver:")
evaluate(my_driver)

print("\nThe learned map from Part 3:")
score = evaluate(map_driver(learned_map))

Both got worse. Rules and learned maps are both tied to the sensor they were built for.

Look through the new camera. The robot is perfectly centered, so the two lines should sit at the same distance left and right of the orange line.

In [ ]:
# What does the new camera see when the robot is perfectly centered?

image = look(offset_cm=0, heading_deg=0)

### The learner's way

Your rules can be repaired: you would have to measure how the camera sits and rebuild your map. That is the extra challenge at the end of this notebook.

The learner does not need to understand the new camera. It needs new examples. Let the instructor drive the new robot, and learn again.

In [ ]:
# Record on the new robot, learn, evaluate. That is all.

X_new, y_new = record_demonstrations()
new_learned_map = learn_map(X_new, y_new)

score = evaluate(map_driver(new_learned_map))

### A third robot

The next robot comes back from repair. Its camera sits straight again. But the cable between camera and computer was plugged in wrong: every pixel arrives in the wrong place. Always the same wrong place, but nobody knows which.

In [ ]:
# Switch to the third robot and look through its camera. The robot sits
# in the center of the lane, pointing along it.

use_third_robot()

image = look(offset_cm=0, heading_deg=0)

In [ ]:
# How do the old drivers do with this camera?

print("The Challenge 1 sample solution:")
evaluate(sample_driver)

print("\nThe learned map from Part 3:")
score = evaluate(map_driver(learned_map))

Before you run the next cell, think:

1. Could you repair your rules for this camera? What would you have to find out first, and how long would that take?
2. Does the learner need to know how the cable was plugged in?

In [ ]:
# Record on the third robot, learn, evaluate.

X_third, y_third = record_demonstrations()
third_learned_map = learn_map(X_third, y_third)

score = evaluate(map_driver(third_learned_map))

show_maps({
    "learned in Part 3": learned_map,
    "learned, third robot": third_learned_map,
})

Compare this score with the learned map in Part 3. It is the same, on every track.

The new map looks like noise to you, because you think in lines, lanes and centers. The learner never did. For it, a pixel is a number with a weight, and it does not matter where that pixel sits in the picture.

## What you built

Two drivers with exactly the same form, pixels times weights, the same machine Rosenblatt built in 1958. The difference was only where the weights came from.

Your hand-written weights came from logic. You had to understand the camera, the lane and the failures, and every new situation, a dashed line, glare, a crooked camera, cost you another round of thinking. In return, you can explain every single weight. And when the picture stopped making sense to you, on the third robot, your logic had nothing left to hold on to.

The learned weights came from examples. The learner understood nothing, needed seconds, and adapted to each new robot by simply watching again. In return, its map is a pile of numbers that no one designed, and it can only be as good as the examples it saw.

Two things about learning you saw with your own eyes:

- **Fitting is not the goal.** A map can match every recorded example perfectly and still drive off the road. What counts is how it does on images it has never seen.
- **The data is the design.** Which tracks, how many laps, how much wobble: with the same learner, these choices made the difference between a crash and your best score.

So learning did not remove human design. It moved it from the rules to the data.

ALVINN, from the lecture, did exactly this in 1989: a camera image, a learned set of weights with one small hidden layer in between, and a human driver as the instructor.

## Extra Challenge: Repair the Rules

If you have time left: repair your hand-written driver for the second robot, the one with the crooked camera. Like a real engineer, you have to find out how the camera is mounted by looking through it.

The rules:

- your driver is a function `my_new_driver(image)`, the same as before
- do not read the mounting numbers from the code above, use the camera
- `evaluate` works as before, now on the second robot

Scores to beat:

| driver | score |
|---|---|
| the Challenge 1 sample solution, unchanged | about 15 |
| the sample solution below | about 8 |
| a really good fix | under 6 |
| the learner, after watching again | about 3 |

In [ ]:
# Back to the second robot. What does its camera see when the robot is
# perfectly centered?

use_new_robot()

image = look(offset_cm=0, heading_deg=0)

In [ ]:
# Extra challenge. Your driver for the second robot. Start from your
# Challenge 1 driver and fix it.

def my_new_driver(image):
    return my_driver(image)


score = evaluate(my_new_driver)

### Hints

1. **Measure first.** The robot above is perfectly centered, so the lines should sit at +30 cm and -30 cm in every row. Where are they in the near rows? Where are they in the far rows?
2. **Two numbers describe the mount.** The true sideways position of every pixel is roughly `LATERAL + a + b * FORWARD`. The shift `a` moves all rows the same way, the turn `b` moves far rows more than near rows. Find `a` and `b` from your measurements.
3. **Reuse your logic.** Build a corrected `TRUE_LATERAL` and use it everywhere your Challenge 1 driver used `LATERAL`. Then tune the gain again.

### A sample solution

Below is the Challenge 1 sample solution, rebuilt on corrected pixel positions. It scores about 8. Look at it if you are stuck, or after your own attempt.

In [ ]:
# Sample solution. Measured with look(): in the near rows the lines appear
# about 12 cm too far right, so a = 0.12. Far rows are shifted about 15 cm
# less per meter of distance, so b = -0.15.

TRUE_LATERAL = LATERAL + 0.12 - 0.15 * FORWARD
sample_new_map = 2.0 * (TRUE_LATERAL - 0.3 * np.sign(TRUE_LATERAL))


def sample_new_driver(image):
    return steer_from_map(image, sample_new_map)


score = evaluate(sample_new_driver)

show_maps({
    "sample solution, second robot": sample_new_map,
    "learned, second robot": new_learned_map,
})